In [ ]:
# # 03 — Forward labels and chronological validation splits

# ## Objective

# Construct forward EURUSD mid-price changes over 1, 5, 30, and 60 seconds,
# then define chronological training, validation, and pilot holdout periods.

# A label is:

#     (mid-price at target time − current mid-price) / 0.0001

# Both endpoints use market states constructed from quotes observed at or
# before their respective timestamps.

# ## Existing policy

# - Maximum snapshot age: 30 seconds, inclusive.
# - Substantial source gap: strictly greater than 30 seconds.
# - Volatility remains calculated from one-second returns.
# - Wide spreads are not automatically excluded.

# ## Important distinctions

# A valid price state may still contain a carried-forward snapshot.
# A zero label can therefore mean that the same source snapshot supplied
# both endpoints. We record that condition explicitly.

# A label requires valid endpoints and an uninterrupted valid sampled price
# path between them. An invalid quote or substantial source gap can break
# that path even when both endpoints look acceptable.

# January is a development pilot. Its holdout is not the project's final
# full-history test period.

# ## Boundary context

# January's last observations need up to 60 seconds of subsequent data.
# A clearly marked cell rescans the existing 2016 CSV with bounded memory,
# retaining only a few minutes around the January boundary.

# The existing January dataset is preserved. Future context is used only
# to construct targets; it is not added to current features.

# ## Execution

# Run from a fresh kernel. If a reusable module needs correction, restart
# the kernel after applying the correction and follow the stated rerun
# instructions.

# All generated datasets use separate generation directories and manifests.
# Original CSVs are never overwritten.

In [ ]:
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research.data import ingest_quotes

NOTEBOOK = "03_labels_and_validation_splits.ipynb"
notebook_started = time.perf_counter()

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (
        path for path in candidates
        if (path / "configs/research_config.example.json").is_file()
        and (path / "src/eurusd_research/features.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open Jupyter inside the project repository.")

config = load_config(REPO_ROOT)
settings = config["sampling_features"]

if settings["max_staleness_seconds"] != 30:
    raise RuntimeError("The confirmed 30-second staleness policy is not active.")
if settings["substantial_gap_seconds"] != 30:
    raise RuntimeError("Expected the existing 30-second substantial-gap threshold.")

base_manifest_path = (
    REPO_ROOT / config["active_features_manifest"]
).resolve()
base_features = json.loads(base_manifest_path.read_text(encoding="utf-8"))

if base_features["status"] != "complete":
    raise RuntimeError("The active feature generation is incomplete.")
if base_features["identity"]["settings"] != settings:
    raise RuntimeError("Configuration differs from the active feature generation.")

START = feature_io._utc(base_features["identity"]["start_utc"])
END = feature_io._utc(base_features["identity"]["end_utc"])

label_settings = config.get("labels_validation", {
    "horizons_seconds": [1, 5, 30, 60],
    "block_hours": 6,
    "max_rows_per_block": 100000,
    "pilot_training_fraction": 0.6,
    "pilot_validation_fraction": 0.2,
})

HORIZONS = label_settings["horizons_seconds"]
if HORIZONS != [1, 5, 30, 60]:
    raise RuntimeError("This notebook expects horizons [1, 5, 30, 60].")

new_sources = {}

print("NOTEBOOK_03_PREFLIGHT")
print("Python:", platform.python_version())
print("Repository:", REPO_ROOT)
print("Feature generation:", base_features["job_id"])
print("Research range:", START.isoformat(), "to", END.isoformat(), "(exclusive)")
print("Maximum snapshot age:", settings["max_staleness_seconds"], "seconds")
print("Volatility return interval: 1 second")
print("Installed packages:", {
    name: version(name)
    for name in ("numpy", "pandas", "pyarrow", "duckdb", "pytest")
})

for relative in (
    "src/eurusd_research/labels.py",
    "src/eurusd_research/validation.py",
    "tests/test_labels.py",
    "tests/test_validation.py",
):
    path = REPO_ROOT / relative
    print(relative, "EXISTS" if path.exists() else "MISSING")

In [ ]:
# ## Chronological split rules

# We will choose pilot dates from observed daily price coverage, without
# looking at future returns.

# Among UTC dates containing usable prices:

# - approximately the first 60% supply training;
# - approximately the next 20% supply validation;
# - the remaining dates supply a pilot holdout.

# Partial trading days count as dates. These fractions are a transparent
# pilot convention, not an optimized selection.

# A row is purged when its target timestamp reaches or crosses the end of
# its assigned period. For example, a training signal whose 60-second target
# lands exactly at validation's start is excluded from training.

# Different horizons can therefore have different eligible training rows.

# The following cell prepares validation.py. It is not written until the
# later file-creation cell has checked every destination.

In [ ]:
new_sources["src/eurusd_research/validation.py"] = r'''
"""Chronological pilot plans and horizon-specific label purging."""

import numpy as np
import pandas as pd


def utc(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is None:
        raise ValueError("A valid timezone-aware timestamp is required.")
    value = value.tz_convert("UTC")
    if value.microsecond or value.nanosecond:
        raise ValueError("Split boundaries must use whole seconds.")
    return value


def validate_plan(plan):
    periods = plan["periods"]
    if [item["role"] for item in periods] != [
        "train", "validation", "pilot_holdout"
    ]:
        raise ValueError("Expected train, validation, and pilot_holdout.")
    previous_end = None
    for item in periods:
        start, end = utc(item["start_utc"]), utc(item["end_utc"])
        if start >= end or (previous_end is not None and start != previous_end):
            raise ValueError("Split periods must be positive and contiguous.")
        previous_end = end
    if plan.get("project_final_test") is not None:
        raise ValueError("This pilot does not define the full-history final test.")
    if not plan.get("feature_job_id"):
        raise ValueError("The plan must identify its feature generation.")
    return plan


def make_pilot_plan(daily, start, end, feature_job_id,
                    training_fraction=0.6, validation_fraction=0.2):
    start, end = utc(start), utc(end)
    if not (
        0 < training_fraction < 1
        and 0 < validation_fraction < 1
        and training_fraction + validation_fraction < 1
    ):
        raise ValueError("Invalid pilot fractions.")

    dates = pd.to_datetime(
        daily.loc[daily["usable_price_rows"].gt(0), "date_utc"], utc=True
    ).drop_duplicates().sort_values().tolist()
    if len(dates) < 10:
        raise ValueError("At least 10 dates with usable prices are required.")
    if any(date < start.normalize() or date >= end for date in dates):
        raise ValueError("Daily coverage lies outside the requested research range.")

    first_cut = int(np.floor(len(dates) * training_fraction))
    second_cut = int(np.floor(
        len(dates) * (training_fraction + validation_fraction)
    ))
    if min(first_cut, second_cut - first_cut, len(dates) - second_cut) < 2:
        raise ValueError("Each pilot segment needs at least two active dates.")

    boundaries = [start, dates[first_cut], dates[second_cut], end]
    roles = ["train", "validation", "pilot_holdout"]
    plan = {
        "stage": "january_development_pilot",
        "feature_job_id": feature_job_id,
        "selection_rule": "chronological fractions of UTC dates with usable prices",
        "training_fraction": training_fraction,
        "validation_fraction": validation_fraction,
        "active_utc_dates": len(dates),
        "project_final_test": None,
        "periods": [
            {
                "role": role,
                "start_utc": boundaries[i].isoformat(),
                "end_utc": boundaries[i + 1].isoformat(),
            }
            for i, role in enumerate(roles)
        ],
    }
    return validate_plan(plan)


def assign_split(labels, plan, horizon):
    validate_plan(plan)
    timestamp = labels["feature_timestamp_utc"]
    target = labels[f"target_timestamp_utc_{horizon}s"]
    valid = labels[f"label_valid_{horizon}s"]
    result = pd.Series("invalid_label", index=labels.index, dtype="string")
    result.loc[valid] = "outside_plan"

    for period in plan["periods"]:
        start, end = utc(period["start_utc"]), utc(period["end_utc"])
        inside = valid & timestamp.ge(start) & timestamp.lt(end)
        result.loc[inside] = period["role"]
        result.loc[inside & target.ge(end)] = "purged_" + period["role"]

    return result
'''.lstrip()

print("Prepared validation.py")

In [ ]:
# ## Forward labels

# For each signal timestamp t and horizon h, use the sampled state at
# exactly t + h. We do not search for the first later quote.

# The label is valid only when:

# 1. The current sampled price is valid.
# 2. The exact target grid timestamp exists.
# 3. The target sampled price is valid and no older than 30 seconds.
# 4. Every sampled price step in (t, t+h] has an intact price path.

# The price_path_break field comes from Notebook 02. It also captures
# invalid raw prices or substantial source gaps observed between grid
# timestamps.

# "Same source record" compares source-file content identity and source
# row number. Timestamp equality alone would be insufficient because
# legitimate updates can share timestamps.

# Future target columns are outcomes and diagnostics. They must never be
# included in model predictors.

In [ ]:
new_sources["src/eurusd_research/labels.py"] = r'''
"""Exact-horizon forward labels and bounded, resumable label storage."""

import hashlib
import json
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from . import features as io
from .data import _sha256, _writer_lock
from .validation import assign_split, utc, validate_plan


INPUT_COLUMNS = [
    "feature_timestamp_utc", "source_timestamp_utc",
    "source_file", "source_sha256", "source_row",
    "snapshot_staleness_seconds", "mid_price",
    "price_valid", "size_pair_valid", "price_path_break",
]


def make_forward_labels(frame, start, end, horizons, pip_size, max_age):
    start, end = utc(start), utc(end)
    if start >= end:
        raise ValueError("The output interval must have positive length.")
    if (
        not horizons
        or any(type(h) is not int or h < 1 for h in horizons)
        or len(set(horizons)) != len(horizons)
    ):
        raise ValueError("Horizons must be unique positive integer seconds.")
    if not np.isfinite(pip_size) or pip_size <= 0:
        raise ValueError("pip_size must be finite and positive.")
    if not np.isfinite(max_age) or max_age < 0:
        raise ValueError("max_age must be finite and nonnegative.")

    q = frame[INPUT_COLUMNS].copy().reset_index(drop=True)
    for name in ("feature_timestamp_utc", "source_timestamp_utc"):
        if not isinstance(q[name].dtype, pd.DatetimeTZDtype):
            raise ValueError(f"{name} must explicitly carry a timezone.")
        q[name] = q[name].dt.tz_convert("UTC").astype("datetime64[us, UTC]")
    for name in ("source_file", "source_sha256"):
        q[name] = q[name].astype("string")
    q["source_row"] = q["source_row"].astype("Int64")
    q["mid_price"] = q["mid_price"].astype("float64")

    for name in ("price_valid", "size_pair_valid", "price_path_break"):
        if q[name].isna().any() or not pd.api.types.is_bool_dtype(q[name]):
            raise ValueError(f"{name} must contain nonmissing booleans.")

    times = pd.DatetimeIndex(q["feature_timestamp_utc"])
    if (
        len(times) == 0 or times.hasnans or not times.is_unique
        or not times.is_monotonic_increasing
        or not times.to_series().diff().iloc[1:].eq(
            pd.Timedelta(seconds=1)
        ).all()
    ):
        raise ValueError("Input must be a complete, unique one-second UTC grid.")

    age = (q["feature_timestamp_utc"] - q["source_timestamp_utc"]).dt.total_seconds()
    if age.lt(0).any():
        raise ValueError("A future source snapshot entered the feature state.")
    if not np.allclose(
        age.to_numpy(dtype=float, na_value=np.nan),
        q["snapshot_staleness_seconds"].to_numpy(dtype=float, na_value=np.nan),
        equal_nan=True,
    ):
        raise ValueError("Stored snapshot ages disagree with source timestamps.")

    valid_state = (
        np.isfinite(q["mid_price"]) & q["mid_price"].gt(0)
        & age.between(0, max_age)
        & q["source_sha256"].notna() & q["source_row"].notna()
    )
    if (q["price_valid"] & ~valid_state).any():
        raise ValueError("A state marked price_valid violates the endpoint policy.")

    out = q.copy()
    prefix = np.r_[0, np.cumsum(q["price_path_break"].to_numpy(dtype=np.int64))]
    current_index = np.arange(len(q))

    for h in horizons:
        wanted = times + pd.Timedelta(seconds=h)
        positions = times.get_indexer(wanted)
        exists = positions >= 0
        target = q.reindex(positions).reset_index(drop=True)
        target_age = (
            pd.Series(wanted) - target["source_timestamp_utc"]
        ).dt.total_seconds()
        target_ok = (
            target["price_valid"].eq(True)
            & np.isfinite(target["mid_price"])
            & target["mid_price"].gt(0)
            & target_age.between(0, max_age)
        ).fillna(False)

        path_ok = np.zeros(len(q), dtype=bool)
        path_ok[exists] = (
            prefix[positions[exists] + 1] - prefix[current_index[exists] + 1]
        ) == 0
        change = (target["mid_price"] - q["mid_price"]) / pip_size
        valid = (
            q["price_valid"] & exists & target_ok
            & path_ok & np.isfinite(change)
        )

        suffix = f"{h}s"
        out[f"target_timestamp_utc_{suffix}"] = pd.Series(wanted)
        out[f"target_source_timestamp_utc_{suffix}"] = target["source_timestamp_utc"]
        out[f"target_source_sha256_{suffix}"] = target["source_sha256"].astype("string")
        out[f"target_source_row_{suffix}"] = target["source_row"].astype("Int64")
        out[f"target_snapshot_age_seconds_{suffix}"] = target_age
        out[f"target_mid_price_{suffix}"] = target["mid_price"]
        out[f"label_{suffix}_pips"] = change.where(valid)
        out[f"label_valid_{suffix}"] = valid.astype(bool)
        out[f"label_reason_{suffix}"] = pd.Series(np.select(
            [
                ~q["price_valid"].to_numpy(),
                ~exists,
                ~target_ok.to_numpy(dtype=bool),
                ~path_ok,
                ~np.isfinite(change.to_numpy()),
            ],
            [
                "invalid_feature_state", "missing_target",
                "invalid_target_state", "broken_price_path", "nonfinite_label",
            ],
            default="valid",
        ), dtype="string")
        out[f"same_source_record_{suffix}"] = (
            valid
            & q["source_sha256"].eq(target["source_sha256"])
            & q["source_row"].eq(target["source_row"])
        ).fillna(False).astype(bool)

    out = out.loc[
        out["feature_timestamp_utc"].ge(start)
        & out["feature_timestamp_utc"].lt(end)
    ].reset_index(drop=True)
    if len(out) != int((end - start).total_seconds()):
        raise ValueError("The requested output range is not fully represented.")
    return out
'''.lstrip()

print("Prepared exact-horizon label calculations")

In [ ]:
# ## Storage, provenance, and reruns

# Labels will be written in six-hour blocks. Each block loads only its
# current grid and the following 60 seconds.

# A generation identity includes input manifest checksums, configuration,
# split dates, implementation checksums, and installed package versions.

# Completed blocks are verified before reuse. Different settings produce
# a different generation; existing generations remain available.

# The storage code reuses Notebook 02's atomic writers, resource limits,
# and Parquet integrity checks. This avoids maintaining a second set of
# file-writing infrastructure.

# Label outputs retain all grid rows, including invalid labels and purge
# statuses. Exclusions are represented explicitly rather than silently
# deleting observations.

In [ ]:
new_sources["src/eurusd_research/labels.py"] += r'''


def _counts(series):
    return {str(key): int(value) for key, value in series.value_counts().items()}


def _statistics(frame, horizons):
    return {
        "rows": len(frame),
        "price_valid_rows": int(frame["price_valid"].sum()),
        "horizons": {
            str(h): {
                "valid_rows": int(frame[f"label_valid_{h}s"].sum()),
                "same_source_record_rows": int(frame[f"same_source_record_{h}s"].sum()),
                "reasons": _counts(frame[f"label_reason_{h}s"]),
                "splits": _counts(frame[f"split_{h}s"]),
            }
            for h in horizons
        },
    }


def build_label_dataset(feature_manifests, root, start, end, settings, plan):
    root = Path(root).resolve()
    start, end = utc(start), utc(end)
    validate_plan(plan)
    paths = [Path(path).resolve() for path in feature_manifests]
    if len(paths) != 2:
        raise ValueError("Expected the research and subsequent-context manifests.")
    sources = [json.loads(path.read_text(encoding="utf-8")) for path in paths]
    base, tail = sources
    horizons = settings["horizons_seconds"]
    feature_settings = base["identity"]["settings"]

    if base["job_id"] != plan["feature_job_id"]:
        raise ValueError("The split plan belongs to a different feature generation.")
    if tail["identity"]["settings"] != feature_settings:
        raise ValueError("Research and context feature policies differ.")
    base_start, base_end = (
        utc(base["identity"]["start_utc"]), utc(base["identity"]["end_utc"])
    )
    if not (
        base_start <= start < end <= base_end
        and utc(tail["identity"]["start_utc"]) == base_end
        and utc(tail["identity"]["end_utc"]) >= end + pd.Timedelta(seconds=max(horizons))
    ):
        raise ValueError("Research or future target coverage is insufficient.")

    for source in sources:
        if source["status"] != "complete":
            raise ValueError("Feature generations must be complete.")
        io._verify_feature_parts(source, io._blocks(
            source["identity"]["start_utc"], source["identity"]["end_utc"],
            source["identity"]["settings"]["block_hours"],
        ))

    hashes = [_sha256(path) for path in paths]
    implementation = hashlib.sha256(b"".join(
        Path(__file__).with_name(name).read_bytes()
        for name in ("labels.py", "validation.py", "features.py")
    )).hexdigest()
    identity = {
        "input_manifest_sha256": hashes,
        "input_job_ids": [source["job_id"] for source in sources],
        "start_utc": start.isoformat(), "end_utc": end.isoformat(),
        "settings": settings, "plan": plan,
        "implementation_sha256": implementation,
        "python": platform.python_version(),
        "versions": {
            name: version(name) for name in ("numpy", "pandas", "pyarrow", "duckdb")
        },
    }
    job_id = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode()
    ).hexdigest()[:24]
    output = root / "data/processed/labels" / job_id
    manifest_root = root / "artifacts/manifests/labels"
    manifest = manifest_root / f"job-{job_id}.json"
    blocks = io._blocks(start, end, settings["block_hours"])

    with _writer_lock(manifest_root):
        if manifest.exists():
            state = json.loads(manifest.read_text(encoding="utf-8"))
            if state["identity"] != identity:
                raise RuntimeError("Existing label generation is incompatible.")
            io._verify_feature_parts(state, blocks)
            if state["status"] == "complete":
                return {**state, "cache_hit": True}
        else:
            output.mkdir(parents=True, exist_ok=True)
            state = {
                "kind": "label_dataset", "job_id": job_id, "status": "running",
                "identity": identity,
                "input_manifests": [str(path) for path in paths],
                "output_dir": str(output), "manifest_path": str(manifest),
                "parts": [],
            }
            io.write_json(manifest, state)

        files = [
            str(Path(source["output_dir"]) / part["path"])
            for source in sources for part in source["parts"]
        ]
        connection = io._connect(files, "features", output, feature_settings)
        try:
            for number, (left, right) in enumerate(blocks):
                if number < len(state["parts"]):
                    continue
                future_end = right + pd.Timedelta(seconds=max(horizons))
                frame = connection.execute(f"""
                    SELECT {", ".join(INPUT_COLUMNS)} FROM features
                    WHERE feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
                      AND feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
                    ORDER BY feature_timestamp_utc
                    LIMIT ?
                """, [
                    left.isoformat(), future_end.isoformat(),
                    settings["max_rows_per_block"] + 1,
                ]).fetchdf()
                if len(frame) > settings["max_rows_per_block"]:
                    raise RuntimeError("Label input exceeded its configured row limit.")
                if len(frame) != int((future_end - left).total_seconds()):
                    raise RuntimeError("A label block lacks complete target context.")

                result = make_forward_labels(
                    frame, left, right, horizons,
                    feature_settings["pip_size"],
                    feature_settings["max_staleness_seconds"],
                )
                for h in horizons:
                    result[f"split_{h}s"] = assign_split(result, plan, h)

                relative = (
                    f"year={left.year:04d}/month={left.month:02d}/"
                    f"part-{left.strftime('%Y%m%dT%H%M%S')}.parquet"
                )
                record = io._save_frame(result, output / relative)
                if state["parts"] and (
                    record["schema_sha256"] != state["parts"][0]["schema_sha256"]
                ):
                    raise RuntimeError("Label schema changed between blocks.")
                record.update({
                    "path": relative,
                    "start_utc": left.isoformat(), "end_utc": right.isoformat(),
                    "statistics": _statistics(result, horizons),
                    "input_rows_collected": len(frame),
                    "observed_frame_bytes": int(
                        frame.memory_usage(index=True, deep=True).sum()
                        + result.memory_usage(index=True, deep=True).sum()
                    ),
                })
                state["parts"].append(record)
                io.write_json(manifest, state)
                if (number + 1) % 8 == 0 or number + 1 == len(blocks):
                    print(f"Label blocks committed: {number + 1}/{len(blocks)}")
        finally:
            connection.close()

        if [_sha256(path) for path in paths] != hashes:
            raise RuntimeError("An input manifest changed during label construction.")
        for source in sources:
            io._verify_feature_parts(source, io._blocks(
                source["identity"]["start_utc"], source["identity"]["end_utc"],
                source["identity"]["settings"]["block_hours"],
            ))
        state["status"] = "complete"
        io._verify_feature_parts(state, blocks)
        io.write_json(manifest, state)
        return {**state, "cache_hit": False}


def load_label_rows(state, start, end, feature_settings, maximum_rows=100000):
    files = [
        str(Path(state["output_dir"]) / part["path"]) for part in state["parts"]
    ]
    connection = io._connect(files, "labels", state["output_dir"], feature_settings)
    try:
        frame = connection.execute("""
            SELECT * FROM labels
            WHERE feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
              AND feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
            ORDER BY feature_timestamp_utc LIMIT ?
        """, [
            utc(start).isoformat(), utc(end).isoformat(), maximum_rows + 1
        ]).fetchdf()
        if len(frame) > maximum_rows:
            raise RuntimeError("Requested label excerpt exceeds its row limit.")
        for name in frame:
            if "timestamp_utc" in name:
                frame[name] = pd.to_datetime(frame[name], utc=True).astype(
                    "datetime64[us, UTC]"
                )
        return frame
    finally:
        connection.close()
'''

print("Prepared bounded label storage")

In [ ]:
# ## Correctness tests

# These tests use small constructed examples with known answers.

# They check pip arithmetic, exact target matching, carried-forward
# snapshots, stale targets, interrupted paths, future-source rejection,
# processing boundaries, and chronological purging.

# Passing these tests supports implementation correctness. It does not
# establish predictive value.

In [ ]:
new_sources["tests/test_labels.py"] = r'''
import numpy as np
import pandas as pd
import pytest

from eurusd_research.labels import make_forward_labels


def states(n=80, start="2020-01-31T23:59:30Z"):
    times = pd.date_range(start, periods=n, freq="s")
    return pd.DataFrame({
        "feature_timestamp_utc": times,
        "source_timestamp_utc": times,
        "source_file": ["synthetic.csv"] * n,
        "source_sha256": ["synthetic-content"] * n,
        "source_row": pd.Series(np.arange(n), dtype="Int64"),
        "snapshot_staleness_seconds": np.zeros(n),
        "mid_price": 1.1 + np.arange(n) * 0.0001,
        "price_valid": np.ones(n, dtype=bool),
        "size_pair_valid": np.ones(n, dtype=bool),
        "price_path_break": np.r_[True, np.zeros(n - 1, dtype=bool)],
    })


def run(q, start=None, end=None, horizons=(1, 5, 30, 60)):
    start = q["feature_timestamp_utc"].iloc[0] if start is None else start
    end = q["feature_timestamp_utc"].iloc[-1] + pd.Timedelta(seconds=1) if end is None else end
    return make_forward_labels(q, start, end, list(horizons), 0.0001, 30)


def test_exact_horizon_and_pips():
    result = run(states())
    for h in (1, 5, 30, 60):
        assert result.loc[0, f"label_{h}s_pips"] == pytest.approx(h)
        assert (
            result.loc[0, f"target_timestamp_utc_{h}s"]
            - result.loc[0, "feature_timestamp_utc"]
        ).total_seconds() == h


def test_carried_snapshot_does_not_use_next_quote():
    q = states(4)
    q.loc[1, "source_timestamp_utc"] = q.loc[0, "source_timestamp_utc"]
    q.loc[1, "source_row"] = q.loc[0, "source_row"]
    q.loc[1, "mid_price"] = q.loc[0, "mid_price"]
    q.loc[1, "snapshot_staleness_seconds"] = 1.0
    result = run(q, horizons=(1,))
    assert result.loc[0, "label_1s_pips"] == 0
    assert result.loc[0, "same_source_record_1s"]


def test_same_timestamp_different_record_is_not_same_record():
    q = states(3)
    q.loc[1, "source_timestamp_utc"] = q.loc[0, "source_timestamp_utc"]
    q.loc[1, "snapshot_staleness_seconds"] = 1.0
    result = run(q, horizons=(1,))
    assert not result.loc[0, "same_source_record_1s"]


def test_missing_target():
    result = run(states(3), horizons=(5,))
    assert not result["label_valid_5s"].any()
    assert result["label_reason_5s"].eq("missing_target").all()


def test_stale_target_rejected():
    q = states(3)
    q.loc[1, "source_timestamp_utc"] = (
        q.loc[1, "feature_timestamp_utc"] - pd.Timedelta(seconds=31)
    )
    q.loc[1, "snapshot_staleness_seconds"] = 31.0
    q.loc[1, "price_valid"] = False
    result = run(q, horizons=(1,))
    assert not result.loc[0, "label_valid_1s"]
    assert result.loc[0, "label_reason_1s"] == "invalid_target_state"


def test_break_between_valid_endpoints():
    q = states(8)
    q.loc[3, "price_path_break"] = True
    result = run(q, horizons=(5,))
    assert q.loc[0, "price_valid"] and q.loc[5, "price_valid"]
    assert result.loc[0, "label_reason_5s"] == "broken_price_path"
    # A break entering t does not invalidate an otherwise intact future interval.
    assert result.loc[3, "label_reason_1s"] == "valid" if "label_reason_1s" in result else True


def test_new_state_after_break_can_start_new_label():
    q = states(5)
    q.loc[2, "price_path_break"] = True
    result = run(q, horizons=(1,))
    assert not result.loc[1, "label_valid_1s"]
    assert result.loc[2, "label_valid_1s"]


def test_future_source_and_missing_grid_rejected():
    q = states(5)
    q.loc[1, "source_timestamp_utc"] += pd.Timedelta(seconds=1)
    with pytest.raises(ValueError, match="future"):
        run(q)
    with pytest.raises(ValueError, match="one-second"):
        run(states(5).drop(index=2))


def test_partition_and_future_append_consistency():
    q = states(180)
    start = q["feature_timestamp_utc"].iloc[0]
    middle, end = start + pd.Timedelta(seconds=40), start + pd.Timedelta(seconds=100)
    whole = run(q, start, end)
    left = run(q.iloc[:100], start, middle)
    right = run(q.iloc[40:160], middle, end)
    combined = pd.concat([left, right], ignore_index=True)
    pd.testing.assert_frame_equal(whole, combined)

    before = run(q.iloc[:160], start, end)
    after = run(q, start, end)
    pd.testing.assert_frame_equal(before, after)
'''.lstrip()

new_sources["tests/test_validation.py"] = r'''
import numpy as np
import pandas as pd
import pytest

from eurusd_research.validation import (
    assign_split, make_pilot_plan, validate_plan,
)


def plan():
    return {
        "feature_job_id": "synthetic",
        "project_final_test": None,
        "periods": [
            {"role": "train", "start_utc": "2020-01-01T00:00:00Z", "end_utc": "2020-01-02T00:00:00Z"},
            {"role": "validation", "start_utc": "2020-01-02T00:00:00Z", "end_utc": "2020-01-03T00:00:00Z"},
            {"role": "pilot_holdout", "start_utc": "2020-01-03T00:00:00Z", "end_utc": "2020-01-04T00:00:00Z"},
        ],
    }


def test_boundary_equality_is_purged():
    timestamps = pd.to_datetime([
        "2020-01-01T23:59:58Z",
        "2020-01-01T23:59:59Z",
        "2020-01-02T23:59:59Z",
        "2020-01-03T12:00:00Z",
    ], format="ISO8601", utc=True)
    labels = pd.DataFrame({
        "feature_timestamp_utc": timestamps,
        "target_timestamp_utc_1s": timestamps + pd.Timedelta(seconds=1),
        "label_valid_1s": [True, True, True, False],
    })
    assert assign_split(labels, plan(), 1).tolist() == [
        "train", "purged_train", "purged_validation", "invalid_label"
    ]


def test_dates_come_from_observed_coverage():
    dates = pd.date_range("2020-01-01", periods=20, tz="UTC")
    daily = pd.DataFrame({
        "date_utc": dates,
        "usable_price_rows": np.ones(20, dtype=int),
    })
    selected = make_pilot_plan(
        daily, dates[0], dates[-1] + pd.Timedelta(days=1), "synthetic"
    )
    assert pd.Timestamp(selected["periods"][1]["start_utc"]) == dates[12]
    assert pd.Timestamp(selected["periods"][2]["start_utc"]) == dates[16]
    assert selected["project_final_test"] is None


def test_overlapping_and_naive_periods_rejected():
    overlapping = plan()
    overlapping["periods"][1]["start_utc"] = "2020-01-01T12:00:00Z"
    with pytest.raises(ValueError):
        validate_plan(overlapping)

    naive = plan()
    naive["periods"][0]["start_utc"] = "2020-01-01"
    with pytest.raises(ValueError, match="timezone"):
        validate_plan(naive)
'''.lstrip()

# Remove an unnecessary conditional assertion; the separate next test
# explicitly checks that a break entering t does not invalidate (t, t+h].
new_sources["tests/test_labels.py"] = new_sources["tests/test_labels.py"].replace(
    '    # A break entering t does not invalidate an otherwise intact future interval.\n'
    '    assert result.loc[3, "label_reason_1s"] == "valid" if "label_reason_1s" in result else True\n',
    '',
)

print("Prepared focused label and validation tests")

In [ ]:
# ## Create reusable files and verify imports

# The next cell creates the four prepared files.

# Before writing anything, it checks all destinations. Existing identical
# files are reused; different existing contents cause a stop.

# - A notebook explains and orchestrates research.
# - A module contains reusable calculations.
# - A generated output contains execution-specific data or evidence.

# We verify imports in a separate Python process whose working directory
# is notebooks/. This checks the editable installation without adding
# manual sys.path modifications.

In [ ]:
for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if path.exists():
        existing = path.read_text(encoding="utf-8").replace("\r\n", "\n")
        if existing != source:
            raise RuntimeError(
                f"Existing contents differ: {relative}. "
                "Stop and return this message; nothing has been overwritten."
            )

for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(source)
        print("CREATED", relative)
    else:
        print("REUSED", relative)

importlib.invalidate_caches()

probe = subprocess.run(
    [
        sys.executable, "-c",
        "import json; from pathlib import Path; "
        "from eurusd_research import labels, validation; "
        "print(json.dumps({'labels': str(Path(labels.__file__).resolve()), "
        "'validation': str(Path(validation.__file__).resolve())}))",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True,
    text=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("Imports from notebooks/ failed.")

import_paths = json.loads(probe.stdout)
for name, filename in import_paths.items():
    expected = REPO_ROOT / "src/eurusd_research" / f"{name}.py"
    if Path(filename).resolve() != expected.resolve():
        raise RuntimeError(f"Imported the wrong project module: {name}")

from eurusd_research.labels import build_label_dataset, load_label_rows
from eurusd_research.validation import make_pilot_plan, validate_plan

print("PASS: imports resolve to this repository from notebooks/")

In [ ]:
# ## Run correctness tests

# The subprocess uses this Jupyter kernel's Python executable.

# All existing and newly introduced correctness tests run together.
# Continue only if pytest reports success.

In [ ]:
test_run = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "tests/test_alignment.py",
        "tests/test_features.py",
        "tests/test_partition_consistency.py",
        "tests/test_labels.py",
        "tests/test_validation.py",
        "-q", "--color=no",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)

if test_run.returncode:
    raise RuntimeError("Tests failed. Stop and return Cell 14 output.")

test_summary = test_run.stdout.strip().splitlines()[-1]
tests_passed = True

In [ ]:
# ## Choose and freeze pilot dates

# The following query returns one row per UTC date, rather than collecting
# the month into pandas.

# Only current-price availability is used to choose dates. No label
# values or predictive results are inspected.

# The resulting date plan is saved as an immutable generated artifact.
# On a rerun, a different proposed plan causes a stop rather than silently
# moving the holdout boundary.

# The full-history final test remains undefined.

In [ ]:
base_files = [
    str(Path(base_features["output_dir"]) / part["path"])
    for part in base_features["parts"]
]
feature_io._verify_feature_parts(
    base_features,
    feature_io._blocks(START, END, settings["block_hours"]),
)

connection = feature_io._connect(
    base_files, "features",
    REPO_ROOT / "artifacts/manifests/labels", settings,
)
try:
    daily_coverage = connection.execute("""
        SELECT date_trunc('day', feature_timestamp_utc) AS date_utc,
               count(*) AS clock_rows,
               count(*) FILTER (WHERE price_valid) AS usable_price_rows
        FROM features
        GROUP BY 1 ORDER BY 1
    """).fetchdf()
finally:
    connection.close()

plan = make_pilot_plan(
    daily_coverage, START, END, base_features["job_id"],
    label_settings["pilot_training_fraction"],
    label_settings["pilot_validation_fraction"],
)
validate_plan(plan)

plan_path = (
    REPO_ROOT / "artifacts/manifests/labels"
    / f"pilot-plan-{base_features['job_id']}.json"
)
if plan_path.exists():
    saved_plan = json.loads(plan_path.read_text(encoding="utf-8"))
    if saved_plan != plan:
        raise RuntimeError(
            "The existing frozen pilot plan differs. Stop before changing splits."
        )
else:
    feature_io.write_json(plan_path, plan)

print("PILOT_SPLIT_PLAN")
print(json.dumps(plan, indent=2))
print("Calendar dates inspected:", len(daily_coverage))
print("No label values or holdout performance inspected.")

In [ ]:
# ## Explicit boundary-context ingestion

# This cell scans the complete existing 2016 CSV because the source is a
# single large file. Processing remains bounded in memory.

# It retains only:

# - enough preceding quotes to reconstruct the boundary features;
# - a short overlap used to compare against existing January features;
# - the following 60 seconds needed for targets.

# It does not ingest full-history output or rebuild January.

# The overlap comparison checks every existing feature column, including
# volatility and activity. A disagreement stops execution.

# If OneDrive previously locked manifest files, keep syncing paused during
# this operation.

# The six-GiB free-space guard also allows room for later DuckDB spill files.

In [ ]:
if not tests_passed:
    raise RuntimeError("Run the correctness tests first.")
if shutil.disk_usage(REPO_ROOT).free < 6 * 1024**3:
    raise RuntimeError("Free at least 6 GiB before proceeding.")

context_started = time.perf_counter()
overlap_seconds = 30
horizon_seconds = max(HORIZONS)
warmup_seconds = feature_io.lookback_seconds(settings)

context_start = END - pd.Timedelta(
    seconds=warmup_seconds
    + settings["max_staleness_seconds"]
    + overlap_seconds + 1
)
context_end = END + pd.Timedelta(seconds=horizon_seconds)

original_ingestion_path = Path(base_features["input_manifest"])
original_ingestion = json.loads(
    original_ingestion_path.read_text(encoding="utf-8")
)
source_path = Path(original_ingestion["identity"]["source_path"])

print("EXPLICIT CSV SCAN — bounded memory, boundary context only")
print("Source:", source_path.name)
print("Retained range:", context_start.isoformat(), "to", context_end.isoformat())

context_ingestion = ingest_quotes(
    source_path,
    Path(original_ingestion["output_dir"]).parent,
    original_ingestion_path.parent,
    context_start,
    context_end,
    **original_ingestion["identity"]["parameters"],
)

if (
    context_ingestion["identity"]["source_sha256"]
    != original_ingestion["identity"]["source_sha256"]
):
    raise RuntimeError("The original source content has changed.")
if pd.Timestamp(context_ingestion["source_utc_max"]) < context_end:
    raise RuntimeError(
        "The source does not establish coverage beyond the target-context range."
    )

overlap_start = END - pd.Timedelta(seconds=overlap_seconds)
existing_overlap = feature_io.load_feature_rows(
    base_features, overlap_start, END, settings,
)
reconstructed_overlap = feature_io.reference_feature_block(
    context_ingestion, overlap_start, END, settings,
    REPO_ROOT / "artifacts/manifests/labels/context_check",
)
pd.testing.assert_frame_equal(
    existing_overlap.reset_index(drop=True),
    reconstructed_overlap.reset_index(drop=True),
    check_dtype=False, rtol=1e-10, atol=1e-12,
)

tail_features = feature_io.build_feature_dataset(
    Path(context_ingestion["manifest_path"]),
    REPO_ROOT, END, context_end, settings,
)
tail_manifest_path = Path(tail_features["manifest_path"])

context_runtime = time.perf_counter() - context_started

print("PASS: all feature columns agree across the boundary overlap.")
print("Context quote rows retained:", context_ingestion["rows_in_scope"])
print("Additional feature grid rows:", sum(p["rows"] for p in tail_features["parts"]))
print("Runtime seconds:", round(context_runtime, 3))
print("January feature generation remains:", base_features["job_id"])

In [ ]:
# ## Build labels for one day, then January

# The day is a pilot execution check. The month is the authoritative
# January label dataset.

# Both read the same January feature generation and the same subsequent
# target context. They are never concatenated.

# Their identical overlapping rows will be checked below.

# All grid rows are retained. A missing label is accompanied by a reason;
# a label excluded at a split boundary has an explicit purge status.

In [ ]:
input_manifests = [base_manifest_path, tail_manifest_path]

DAY_START = feature_io._utc(config["ingestion"]["day_start_utc"])
DAY_END = feature_io._utc(config["ingestion"]["day_end_utc"])

if not START <= DAY_START < DAY_END <= END:
    raise RuntimeError("The configured pilot day lies outside the research range.")

started = time.perf_counter()
day_labels = build_label_dataset(
    input_manifests, REPO_ROOT, DAY_START, DAY_END, label_settings, plan,
)
day_runtime = time.perf_counter() - started

started = time.perf_counter()
month_labels = build_label_dataset(
    input_manifests, REPO_ROOT, START, END, label_settings, plan,
)
month_runtime = time.perf_counter() - started

print("DAY_LABELS")
print({
    "rows": sum(p["rows"] for p in day_labels["parts"]),
    "cache_hit": day_labels["cache_hit"],
    "runtime_seconds": round(day_runtime, 3),
})
print("MONTH_LABELS")
print({
    "rows": sum(p["rows"] for p in month_labels["parts"]),
    "cache_hit": month_labels["cache_hit"],
    "runtime_seconds": round(month_runtime, 3),
})

In [ ]:
# ## Verify real-data consistency and label integrity

# We compare the independently generated day against the same timestamps
# inside the month. EXCEPT ALL compares complete rows while retaining
# duplicate multiplicity.

# Additional checks require:

# - one output row per research second;
# - no future source snapshot at either endpoint;
# - exact target timestamps;
# - no stale target marked valid;
# - no missing or nonfinite valid label;
# - no numeric label attached to an invalid label;
# - no retained split row whose target reaches the next period;
# - zero change when both endpoints use the same source record.

# Coverage counts do not represent independent statistical observations.
# No model performance is measured here.

In [ ]:
month_files = [
    str(Path(month_labels["output_dir"]) / p["path"])
    for p in month_labels["parts"]
]
day_files = [
    str(Path(day_labels["output_dir"]) / p["path"])
    for p in day_labels["parts"]
]

connection = feature_io._connect(
    month_files, "month_labels", month_labels["output_dir"], settings,
)
try:
    connection.read_parquet(
        day_files, hive_partitioning=False
    ).create_view("day_labels")

    difference_rows = int(connection.execute("""
        WITH wanted AS (
            SELECT * FROM month_labels
            WHERE feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
              AND feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
        ),
        left_only AS (
            SELECT * FROM day_labels EXCEPT ALL SELECT * FROM wanted
        ),
        right_only AS (
            SELECT * FROM wanted EXCEPT ALL SELECT * FROM day_labels
        )
        SELECT (SELECT count(*) FROM left_only)
             + (SELECT count(*) FROM right_only)
    """, [DAY_START.isoformat(), DAY_END.isoformat()]).fetchone()[0])

    rows, unique_rows, first_time, last_time, future_current = (
        connection.execute("""
            SELECT count(*), count(DISTINCT feature_timestamp_utc),
                   min(feature_timestamp_utc), max(feature_timestamp_utc),
                   count(*) FILTER (
                       WHERE source_timestamp_utc > feature_timestamp_utc
                   )
            FROM month_labels
        """).fetchone()
    )
    assert difference_rows == 0, "Day and month label outputs differ."
    assert rows == unique_rows == int((END - START).total_seconds())
    assert pd.Timestamp(first_time) == START
    assert pd.Timestamp(last_time) == END - pd.Timedelta(seconds=1)
    assert future_current == 0

    horizon_summary = {}
    for h in HORIZONS:
        bad = int(connection.execute(f"""
            SELECT count(*) FROM month_labels
            WHERE target_timestamp_utc_{h}s
                    <> feature_timestamp_utc + INTERVAL '{h} seconds'
               OR target_source_timestamp_utc_{h}s > target_timestamp_utc_{h}s
               OR (
                   label_valid_{h}s AND (
                       label_{h}s_pips IS NULL
                       OR NOT isfinite(label_{h}s_pips)
                       OR target_source_timestamp_utc_{h}s IS NULL
                       OR target_snapshot_age_seconds_{h}s IS NULL
                       OR target_snapshot_age_seconds_{h}s < 0
                       OR target_snapshot_age_seconds_{h}s > 30
                       OR NOT price_valid
                   )
               )
               OR (NOT label_valid_{h}s AND label_{h}s_pips IS NOT NULL)
               OR (
                   same_source_record_{h}s
                   AND abs(label_{h}s_pips) > 0.000000001
               )
        """).fetchone()[0])
        assert bad == 0, f"Label integrity violations at horizon {h}."

        for period in plan["periods"]:
            leaking = int(connection.execute(f"""
                SELECT count(*) FROM month_labels
                WHERE split_{h}s = ?
                  AND target_timestamp_utc_{h}s >= CAST(? AS TIMESTAMPTZ)
            """, [period["role"], period["end_utc"]]).fetchone()[0])
            assert leaking == 0, f"Unpurged boundary at horizon {h}."

        counts = connection.execute(f"""
            SELECT count(*) FILTER (WHERE label_valid_{h}s),
                   count(*) FILTER (WHERE same_source_record_{h}s)
            FROM month_labels
        """).fetchone()
        reasons = dict(connection.execute(f"""
            SELECT label_reason_{h}s, count(*) FROM month_labels
            GROUP BY 1 ORDER BY 1
        """).fetchall())
        splits = dict(connection.execute(f"""
            SELECT split_{h}s, count(*) FROM month_labels
            GROUP BY 1 ORDER BY 1
        """).fetchall())
        assert sum(reasons.values()) == rows
        assert sum(splits.values()) == rows

        horizon_summary[str(h)] = {
            "valid_label_rows": int(counts[0]),
            "same_source_record_rows": int(counts[1]),
            "reasons": {str(k): int(v) for k, v in reasons.items()},
            "splits": {str(k): int(v) for k, v in splits.items()},
            "integrity_violations": bad,
        }
finally:
    connection.close()

print("PASS: day/month difference rows:", difference_rows)
print("PASS: grid conservation, target timing, staleness, and split purging.")
print("LABEL_COVERAGE")
print(json.dumps(horizon_summary, indent=2))

In [ ]:
# ## Save configuration and execution evidence

# The shared example configuration receives portable label settings.

# Machine-specific manifest selections and the frozen pilot dates remain
# in the ignored local configuration. The original January feature
# generation remains active.

# Documentation records executed checks and confirmed policy choices.
# It does not claim owner review of Notebook 03.

# The public methodology description contains no raw quote excerpts.
# Detailed counts and execution paths are saved in reports/private/.

# Generated outputs replace the corresponding generated execution report
# on rerun; original data and earlier dataset generations remain intact.

In [ ]:
def relative_path(path):
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def update_section(path, name, content):
    """Replace one generated Markdown section while preserving other text."""
    begin = f"<!-- {name}:BEGIN -->"
    finish = f"<!-- {name}:END -->"
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    block = f"{begin}\n{content.strip()}\n{finish}"
    if existing.count(begin) != existing.count(finish) or existing.count(begin) > 1:
        raise RuntimeError(f"Malformed generated section in {path.name}")
    if begin in existing:
        left, remainder = existing.split(begin, 1)
        _, right = remainder.split(finish, 1)
        result = left + block + right
    else:
        result = existing.rstrip() + "\n\n" + block + "\n"
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(result)


example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"
example = json.loads(example_path.read_text(encoding="utf-8-sig"))
local = json.loads(local_path.read_text(encoding="utf-8-sig"))

example["labels_validation"] = label_settings

if "labels_validation" in local and local["labels_validation"] != label_settings:
    raise RuntimeError("A conflicting local label configuration needs review.")
local["labels_validation"] = label_settings
local["active_labels_manifest"] = relative_path(month_labels["manifest_path"])
local["label_context_features_manifest"] = relative_path(tail_manifest_path)
local["pilot_validation_plan"] = plan
local["research_range_utc"] = {
    "start_utc": START.isoformat(),
    "end_utc_exclusive": END.isoformat(),
}

feature_io.write_json(example_path, example)
feature_io.write_json(local_path, local)

public_note = f"""
### Notebook 03 — labels and pilot splits

Implemented exact 1-, 5-, 30-, and 60-second forward mid-price changes in
pips, with backward-constructed endpoint states and explicit price-path
validity checks.

Snapshot ages up to 30 seconds are accepted. A source gap strictly greater
than 30 seconds breaks the price path. Wide spreads are not automatically
excluded.

The owner confirmed retention of volatility calculated from one-second
returns. The earlier proposed ten-second-return implementation is superseded.

Same-source-record endpoints are identified separately. These observations
may have zero labels because the sampled snapshot was carried forward.

Pilot split dates are selected chronologically from observed current-price
coverage. Label intervals reaching the next period are purged. January's
pilot holdout is not the full-history final test.

Execution checks: {test_summary}; the independently generated day and month
have identical overlapping label rows. Target timing, source causality,
staleness, grid conservation, and split-purge checks passed.

Notebook 03 owner review remains pending. Full-history processing,
final-test dates, model fitting, predictive evaluation, and sustained
wide-spread episode analysis remain pending.

Results pending execution and validation.
"""

for relative in ("README.md", "reports/data_quality.md"):
    update_section(REPO_ROOT / relative, "NOTEBOOK_03", public_note)

update_section(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK_03",
    f"""
### Notebook 03

- Raymond Fendi confirmed the 30-second snapshot-age policy and retention
  of one-second-return volatility.
- OpenAI Codex supplied label/split implementation, focused tests, and
  notebook instructions.
- Execution reported: {test_summary}.
- Real-data day/month comparison and label integrity checks passed during
  this notebook execution.
- Notebook 03 owner review and understanding checkpoint: pending.
- No model or predictive finding has been established.
""",
)

update_section(
    REPO_ROOT / "reports/feature_dictionary.md",
    "VOLATILITY_DECISION",
    """
### Confirmed volatility specification

Retain realized volatility from one-second mid-price changes over 60-
and 300-second trailing windows. No ten-second-return replacement is active.
""",
)

label_dictionary = """
# Label dictionary

`feature_timestamp_utc`: historical signal timestamp.

`source_timestamp_utc`, `source_sha256`, `source_row`: current snapshot
provenance.

For each horizon h in 1, 5, 30, and 60 seconds:

- `target_timestamp_utc_{h}s`: feature timestamp plus exactly h seconds.
- `target_source_timestamp_utc_{h}s`: snapshot underlying the target state.
- `target_source_sha256_{h}s`, `target_source_row_{h}s`: target provenance.
- `target_snapshot_age_seconds_{h}s`: target time minus target source time.
- `target_mid_price_{h}s`: sampled target mid-price; an outcome diagnostic.
- `label_{h}s_pips`: forward mid-price difference divided by 0.0001;
  missing when invalid.
- `label_valid_{h}s`: both endpoint states and intervening price path pass.
- `label_reason_{h}s`: first applicable failure reason, or valid.
- `same_source_record_{h}s`: valid endpoints use identical source content
  and source row.
- `split_{h}s`: chronological role, purge status, or invalid-label status.

Future target and label fields must never enter model predictors.

The target state uses quotes observed at or before the exact target time.
No first-quote-after-target substitution is used.

Purging excludes labels whose target reaches or crosses their period end.
The pilot holdout is separate from the future full-history final test.

Displayed sizes are not executed trade volumes. This is a research
prototype. Predictive accuracy does not establish achievable trading profits.
"""
update_section(
    REPO_ROOT / "reports/label_dictionary.md",
    "LABEL_DICTIONARY",
    label_dictionary,
)

versions = {
    name: version(name)
    for name in ("numpy", "pandas", "pyarrow", "duckdb", "pytest", "tzdata", "pytz")
}
environment_path = REPO_ROOT / "reports/private/validated_labels_environment.json"
feature_io.write_json(environment_path, {
    "python": platform.python_version(),
    "platform": platform.system(),
    "packages": versions,
    "validation": test_summary,
})

print("Saved shared label settings and local dataset selections.")
print("Updated documentation; owner review remains pending.")

In [ ]:
# ## Interpretation before modeling

# This notebook establishes what a future-price-change observation means
# and where it may be used.

# It does not determine whether any feature predicts that change.

# Thirty-second snapshot eligibility increases usable sampled states, but
# does not create additional quote updates. The same-source-record counts
# help expose carried-forward observations.

# Label eligibility and feature availability are different:

# - a valid label does not guarantee every candidate feature is available;
# - missing volatility does not automatically invalidate a price label;
# - model-specific eligibility will be defined explicitly in Notebook 04.

# Training-only transformations and matched price-only versus
# price-plus-size evaluation will be introduced with the baseline models.

In [ ]:
summary = {
    "notebook": NOTEBOOK,
    "python_version": platform.python_version(),
    "package_versions": versions,
    "configuration": {
        "research_start_utc": START.isoformat(),
        "research_end_utc_exclusive": END.isoformat(),
        "horizons_seconds": HORIZONS,
        "pip_size": settings["pip_size"],
        "maximum_snapshot_age_seconds": settings["max_staleness_seconds"],
        "substantial_gap_seconds": settings["substantial_gap_seconds"],
        "volatility_return_interval_seconds": 1,
        "label_block_hours": label_settings["block_hours"],
        "maximum_input_rows_per_label_block": label_settings["max_rows_per_block"],
        "pilot_plan": plan,
    },
    "inputs": {
        "research_feature_job": base_features["job_id"],
        "research_feature_rows": sum(p["rows"] for p in base_features["parts"]),
        "context_feature_job": tail_features["job_id"],
        "context_feature_rows": sum(p["rows"] for p in tail_features["parts"]),
        "context_quote_rows": context_ingestion["rows_in_scope"],
        "context_source_rows_scanned": context_ingestion["rows_scanned"],
    },
    "outputs": {
        "day_label_rows": sum(p["rows"] for p in day_labels["parts"]),
        "month_label_rows": int(rows),
        "grid_utc_min": pd.Timestamp(first_time).isoformat(),
        "grid_utc_max": pd.Timestamp(last_time).isoformat(),
        "horizons": horizon_summary,
    },
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "boundary_feature_overlap": "passed",
        "day_month_difference_rows": difference_rows,
        "future_current_source_violations": int(future_current),
        "label_integrity_and_split_purging": "passed",
        "holdout_predictive_results_inspected": False,
        "owner_review": "pending",
    },
    "measurements": {
        "context_preparation_seconds": round(context_runtime, 3),
        "day_label_seconds": round(day_runtime, 3),
        "month_label_seconds": round(month_runtime, 3),
        "month_parquet_bytes": sum(p["bytes"] for p in month_labels["parts"]),
        "maximum_input_rows_collected": max(
            p["input_rows_collected"] for p in month_labels["parts"]
        ),
        "maximum_observed_input_output_frame_bytes": max(
            p["observed_frame_bytes"] for p in month_labels["parts"]
        ),
        "day_cache_hit": day_labels["cache_hit"],
        "month_cache_hit": month_labels["cache_hit"],
        "notebook_elapsed_seconds": round(
            time.perf_counter() - notebook_started, 3
        ),
    },
    "warnings": [
        "January is a development pilot; full-history final-test dates remain unset.",
        "A valid endpoint may use a carried-forward snapshot up to 30 seconds old.",
        "Same-source-record counts are descriptive, not independent observations.",
        "Displayed-size denomination remains unresolved.",
        "True side-specific quote ages are unavailable.",
        "Frame memory measurements are not total process peak memory.",
        "Full-history quality and sustained-wide-spread analysis remain pending.",
        "No model, predictive-value, or profitability finding has been established.",
    ],
    "saved_outputs": {
        "active_labels_manifest": relative_path(month_labels["manifest_path"]),
        "research_features_manifest": relative_path(base_manifest_path),
        "context_features_manifest": relative_path(tail_manifest_path),
        "pilot_plan": relative_path(plan_path),
        "label_dictionary": "reports/label_dictionary.md",
        "validated_environment": relative_path(environment_path),
        "execution_summary": "reports/private/notebook03_execution.json",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook03_execution.json", summary
)

print("NOTEBOOK_03_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))

In [ ]:
# ## Review checkpoint

# ### Financial conclusion

# No predictive conclusion exists yet. We have defined forward price changes
# and the conditions under which they are acceptable research observations.

# ### What the code does

# It creates exact-horizon labels, retains source provenance, rejects broken
# price paths, identifies carried-forward endpoints, and assigns purged
# chronological pilot splits.

# ### Validation evidence

# Use the execution summary to confirm test results, boundary consistency,
# row conservation, label integrity, and split purging.

# ### Remaining limitations

# January alone does not establish full-history stability. The 30-second
# snapshot-age policy remains a research assumption. Displayed-size units
# and true side-specific quote ages remain unresolved.

# ### Return these outputs

# 1. The complete NOTEBOOK_03_SUMMARY printed by Cell 26.
# 2. Any warning or error raised before that summary.
# 3. Your answer to the understanding question below.

# ### Understanding question

# Why must we exclude a training observation whose feature timestamp is
# before validation starts, but whose 60-second target lands exactly at
# validation's start?

# Stop here. Do not inspect pilot holdout returns or fit models yet.